# Long Context Evaluation

How well do LLMs handle long documents? This notebook tests: Needle in a Haystack (NIAH), Key Value Retrieval, Position Bias, Multi Hop Reasoning, Counting, Chronological Ordering, Cross Document Comparison, and Summarization.

In [1]:
# !pip install tiktoken sentence-transformers transformers accelerate scikit-learn

In [1]:
GROQ_MODEL_FAST  = "llama-3.1-8b-instant"
GROQ_MODEL_SMART = "llama-3.3-70b-versatile"
OPENAI_BASE_URL = "https://api.groq.com/openai/v1"

import nltk
nltk.download('wordnet', quiet=True)
nltk.download('punkt',   quiet=True)
nltk.download('punkt_tab', quiet=True)

print('✅ All packages installed!')

import os, random, string, uuid, textwrap
import json, re
import numpy as np
from groq import Groq
from dotenv import load_dotenv
load_dotenv()

groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))

def groq_chat(prompt, system="You are a helpful assistant.",
              model=GROQ_MODEL_SMART, temperature=0.0, max_tokens=600):
    resp = groq_client.chat.completions.create(
        model=model,
        messages=[{"role": "system", "content": system}, {"role": "user", "content": prompt}],
        temperature=temperature, max_tokens=max_tokens
    )
    return resp.choices[0].message.content.strip()

def parse_json(raw):
    raw = re.sub(r'```json|```', '', raw).strip()
    try: return json.loads(raw)
    except json.JSONDecodeError:
        m = re.search(r'\{.*\}', raw, re.DOTALL)
        return json.loads(m.group()) if m else {}

def show(title, d):
    print(f"\n{'='*60}\n  📊 {title}\n{'='*60}")
    for k, v in d.items():
        print(f"  {k:<30}: {v:.4f}" if isinstance(v, float) else f"  {k:<30}: {v}")

print('🤖 Groq test:', groq_chat('Reply with only: Groq is ready!', max_tokens=20))

# ── Shared filler paragraphs ─────────────────────────────
FILLER_PARAGRAPHS = [
    'The history of mathematics spans thousands of years. Ancient civilizations including the Babylonians, Egyptians, and Greeks all made significant contributions to mathematical knowledge. The Pythagorean theorem, for example, was known long before Pythagoras formalized it.',
    'Ocean currents play a critical role in regulating global climate patterns. The Gulf Stream carries warm water from the Gulf of Mexico northward along the Atlantic coast, moderating temperatures in Western Europe. Without these currents, weather patterns would differ dramatically.',
    'The development of the printing press in the 15th century transformed European society. Johannes Gutenberg is widely credited with its invention around 1440. Mass production of books led to increased literacy rates and the rapid spread of ideas during the Renaissance.',
    'Photosynthesis is the process by which plants convert sunlight into chemical energy. Chlorophyll molecules absorb light primarily in the blue and red wavelengths, reflecting green light. This process produces oxygen as a byproduct and is essential for life on Earth.',
    'The field of artificial intelligence has evolved significantly since its founding in the 1950s. Early AI research focused on symbolic reasoning and expert systems. Modern approaches rely heavily on deep learning and neural networks trained on massive datasets.',
    'Volcanoes are geological formations created by the eruption of molten rock from beneath the Earths crust. The Ring of Fire, encircling the Pacific Ocean, contains approximately 75 percent of the worlds active volcanoes. Eruptions can have both devastating and beneficial effects.',
    'Classical music has undergone many transformations over the centuries. From the Baroque period of Bach and Handel through the Romantic era of Chopin and Liszt, each epoch introduced new forms, harmonies, and instruments. The modern era continues to push boundaries.',
    'The human immune system is a complex network of cells, tissues, and organs that work together to defend the body against pathogens. White blood cells, including T cells and B cells, identify and neutralize foreign invaders. Vaccination leverages this system to build immunity.',
    'Astronomy has captivated humans since ancient times. The invention of the telescope by Galileo in 1609 revolutionized our understanding of the cosmos. Today, space telescopes like the James Webb observe galaxies billions of light-years away.',
    'Architecture reflects the cultural values and technological capabilities of civilizations. From the pyramids of Giza to modern skyscrapers, building design has always balanced form and function. Sustainable architecture is now a growing priority worldwide.',
]

✅ All packages installed!
🤖 Groq test: Groq is ready!


## 1. Needle-in-a-Haystack (NIAH)
### Tool: Groq + synthetic long context

In [6]:
NEEDLE = 'The secret project codename is Operation Moonlit Falcon and it was initiated on March 15, 2019.'

def build_haystack(n_paragraphs, needle_position):
    paras = [random.choice(FILLER_PARAGRAPHS) for _ in range(n_paragraphs)]
    insert_idx = max(0, min(int(needle_position * len(paras)), len(paras) - 1))
    paras.insert(insert_idx, NEEDLE)
    return '\n\n'.join(paras)

NIAH_QUESTION = 'What is the secret project codename and when was it initiated?'
NIAH_KEYWORDS = ['moonlit falcon', 'march 15', '2019']

def score_niah(answer, keywords=NIAH_KEYWORDS):
    answer_lower = answer.lower()
    hits = [kw for kw in keywords if kw in answer_lower]
    return len(hits) / len(keywords)

positions  = [0.0, 0.25, 0.5, 0.75, 1.0]
ctx_sizes  = [10, 20, 40]
labels     = {0.0: 'START', 0.25: '25%', 0.5: 'MID', 0.75: '75%', 1.0: 'END'}

print('\n🪡 Needle-in-a-Haystack Evaluation')
print('='*75)
results_niah = []
for n_para in ctx_sizes:
    print(f'\n  Context size: ~{n_para} paragraphs')
    for pos in positions:
        haystack = build_haystack(n_para, pos)
        prompt = f'{haystack}\n\nQuestion: {NIAH_QUESTION}\nAnswer concisely.'
        try:
            answer = groq_chat(prompt, model=GROQ_MODEL_FAST, max_tokens=80)
            sc = score_niah(answer)
        except Exception as e:
            answer = f'ERROR: {e}'
            sc = 0.0
        results_niah.append({'paragraphs': n_para, 'position': labels[pos], 'score': sc})
        status = '✅' if sc >= 0.66 else '⚠️' if sc > 0 else '❌'
        print(f'    {status} pos={labels[pos]:<6} score={sc:.2f}  A: {answer[:70]}')

avg = np.mean([r['score'] for r in results_niah])
print(f'\n  Overall NIAH Recall: {avg:.2%}')

# Position heatmap
print('\n  NIAH Heatmap (rows=size, cols=position):')
print(f'    {"":>4}  ' + '  '.join(f'{labels[p]:<6}' for p in positions))
for n in ctx_sizes:
    row_scores = [next((r['score'] for r in results_niah if r['paragraphs']==n and r['position']==labels[p]), 0) for p in positions]
    row_str = '  '.join(f'{"█" if s>=0.66 else "▓" if s>0 else "░"} {s:.2f}' for s in row_scores)
    print(f'    {n:>3}p  {row_str}')


🪡 Needle-in-a-Haystack Evaluation

  Context size: ~10 paragraphs
    ✅ pos=START  score=1.00  A: The secret project codename is Operation Moonlit Falcon, initiated on 
    ✅ pos=25%    score=1.00  A: The secret project codename is Operation Moonlit Falcon, initiated on 
    ✅ pos=MID    score=1.00  A: The secret project codename is Operation Moonlit Falcon, and it was in
    ✅ pos=75%    score=1.00  A: The secret project codename is Operation Moonlit Falcon, and it was in
    ✅ pos=END    score=1.00  A: The secret project codename is Operation Moonlit Falcon, and it was in

  Context size: ~20 paragraphs
    ✅ pos=START  score=1.00  A: The secret project codename is Operation Moonlit Falcon, initiated on 
    ✅ pos=25%    score=1.00  A: The secret project codename is Operation Moonlit Falcon, and it was in
    ✅ pos=MID    score=1.00  A: The secret project codename is Operation Moonlit Falcon, and it was in
    ✅ pos=75%    score=1.00  A: The secret project codename is Operation Moon

## 2. Key-Value Retrieval over Long Context
### Tool: Groq + synthetic KV pairs

In [7]:
def generate_kv_context(n_pairs, target_idx=None):
    keys = [f'item_{uuid.uuid4().hex[:8]}' for _ in range(n_pairs)]
    vals = [f'val_{"".join(random.choices(string.ascii_lowercase, k=6))}' for _ in range(n_pairs)]
    if target_idx is None:
        target_idx = random.randint(0, n_pairs - 1)
    lines = [f'  {keys[i]}: {vals[i]}' for i in range(n_pairs)]
    context = 'Below is a registry of items and their associated values:\n' + '\n'.join(lines)
    return context, keys[target_idx], vals[target_idx], target_idx

kv_sizes = [50, 100, 200]

print('\n🔑 Key-Value Retrieval Evaluation')
print('='*65)
kv_results = []
for n in kv_sizes:
    hits = 0
    trials = 3
    for t in range(trials):
        target_positions = [0, n // 2, n - 1]
        tidx = target_positions[t]
        ctx, tgt_key, tgt_val, _ = generate_kv_context(n, target_idx=tidx)
        prompt = f'{ctx}\n\nWhat is the value associated with the key "{tgt_key}"? Reply with only the value.'
        try:
            answer = groq_chat(prompt, model=GROQ_MODEL_FAST, max_tokens=30)
        except Exception as e:
            answer = f'ERROR: {e}'
        hit = tgt_val.lower() in answer.lower()
        if hit: hits += 1
        pos_label = ['START', 'MID', 'END'][t]
        print(f'  n={n:<4} pos={pos_label:<6} {"✅" if hit else "❌"}  expected={tgt_val}  got={answer[:40]}')
    kv_results.append({'n_pairs': n, 'accuracy': hits / trials})

print(f'\n  Summary:')
for r in kv_results:
    print(f'    {r["n_pairs"]} pairs -> accuracy = {r["accuracy"]:.1%}')


🔑 Key-Value Retrieval Evaluation
  n=50   pos=START  ✅  expected=val_xwmevm  got=val_xwmevm
  n=50   pos=MID    ✅  expected=val_uclfhy  got=val_uclfhy
  n=50   pos=END    ✅  expected=val_ygzlzo  got=val_ygzlzo
  n=100  pos=START  ✅  expected=val_rrkmmx  got=val_rrkmmx
  n=100  pos=MID    ✅  expected=val_jcrqwf  got=val_jcrqwf
  n=100  pos=END    ✅  expected=val_ntuofm  got=val_ntuofm
  n=200  pos=START  ✅  expected=val_jnsrnn  got=val_jnsrnn
  n=200  pos=MID    ✅  expected=val_zvwotc  got=val_zvwotc
  n=200  pos=END    ✅  expected=val_ebvsla  got=val_ebvsla

  Summary:
    50 pairs -> accuracy = 100.0%
    100 pairs -> accuracy = 100.0%
    200 pairs -> accuracy = 100.0%


## 3. Position Bias Detection (Lost-in-the-Middle)
### Tool: Groq + multi-document QA

In [8]:
GOLD_DOC = 'Document: The Zephyr Protocol was developed in 2021 by Dr. Elena Vasquez at the Helios Research Institute. It reduces energy consumption in data centers by 37 percent through adaptive cooling algorithms.'

DISTRACTOR_DOCS = [
    'Document: The Global Seed Vault in Svalbard stores over one million seed samples from around the world to preserve crop diversity for future generations.',
    'Document: The Mariana Trench is the deepest part of the worlds oceans, reaching a depth of approximately 36,000 feet in the Challenger Deep.',
    'Document: Renaissance artists like Leonardo da Vinci and Michelangelo transformed European art with techniques such as perspective and chiaroscuro.',
    'Document: The Trans-Siberian Railway spans 9,289 kilometers from Moscow to Vladivostok, making it the longest railway line in the world.',
    'Document: CRISPR-Cas9 gene editing technology allows scientists to modify DNA sequences with unprecedented precision, opening new avenues in medicine.',
    'Document: The Antarctic ice sheet contains about 26.5 million cubic kilometers of ice, representing approximately 61 percent of all fresh water on Earth.',
    'Document: Blockchain technology was first conceptualized in 2008 as the underlying system for Bitcoin, enabling decentralized digital transactions.',
    'Document: The Voyager 1 spacecraft, launched in 1977, is the most distant human-made object from Earth, currently in interstellar space.',
    'Document: Coral reefs support approximately 25 percent of all marine species despite covering less than one percent of the ocean floor.',
]

QUESTION_POS = 'Who developed the Zephyr Protocol and by what percentage does it reduce energy consumption?'
KEYWORDS_POS = ['elena vasquez', '37']

positions_to_test = [0, 2, 4, 7, 9]

print('\n📍 Position Bias (Lost-in-the-Middle) Evaluation')
print('='*70)
pos_results = []
for gold_pos in positions_to_test:
    docs = DISTRACTOR_DOCS.copy()
    docs.insert(gold_pos, GOLD_DOC)
    ctx = '\n\n'.join(docs)
    prompt = f'Read the following documents and answer the question.\n\n{ctx}\n\nQuestion: {QUESTION_POS}\nAnswer concisely.'
    answer = groq_chat(prompt, model=GROQ_MODEL_FAST, max_tokens=80)
    answer_lower = answer.lower()
    hits = sum(1 for kw in KEYWORDS_POS if kw in answer_lower)
    score = hits / len(KEYWORDS_POS)
    pos_results.append({'position': gold_pos, 'score': score})
    status = '✅' if score >= 0.5 else '❌'
    print(f'  {status} Gold doc at pos {gold_pos}/9  score={score:.2f}  A: {answer[:65]}')

scores_by_pos = {r['position']: r['score'] for r in pos_results}
edge_avg = np.mean([scores_by_pos.get(0, 0), scores_by_pos.get(9, 0)])
mid_avg  = np.mean([scores_by_pos.get(p, 0) for p in [2, 4, 7]])
bias = edge_avg - mid_avg
print(f'\n  Edge positions avg: {edge_avg:.2f}')
print(f'  Middle positions avg: {mid_avg:.2f}')
print(f'  Position bias (edge - middle): {bias:+.2f}  {"⚠️ Biased" if abs(bias) > 0.2 else "✅ Low bias"}')


📍 Position Bias (Lost-in-the-Middle) Evaluation
  ✅ Gold doc at pos 0/9  score=1.00  A: Dr. Elena Vasquez developed the Zephyr Protocol in 2021, which re
  ✅ Gold doc at pos 2/9  score=1.00  A: Dr. Elena Vasquez developed the Zephyr Protocol in 2021. It reduc
  ✅ Gold doc at pos 4/9  score=1.00  A: Dr. Elena Vasquez developed the Zephyr Protocol in 2021. It reduc
  ✅ Gold doc at pos 7/9  score=1.00  A: Dr. Elena Vasquez developed the Zephyr Protocol in 2021. It reduc
  ✅ Gold doc at pos 9/9  score=1.00  A: Dr. Elena Vasquez developed the Zephyr Protocol, which reduces en

  Edge positions avg: 1.00
  Middle positions avg: 1.00
  Position bias (edge - middle): +0.00  ✅ Low bias


## 4. Long-Range Multi-Hop Reasoning
### Tool: Groq + multi-step evidence chains

In [9]:
MULTI_HOP_CASES = [
    {
        'facts': [
            'Professor Alan Mitchell works at the Northbridge Institute of Technology.',
            'The Northbridge Institute of Technology is located in Portland, Oregon.',
            'Professor Mitchell won the Turing Award in 2022.',
        ],
        'question': 'In which city does the 2022 Turing Award winner work?',
        'answer_keywords': ['portland'],
        'hops': 3,
    },
    {
        'facts': [
            'NovaChem Inc. produces a compound called Seralyte.',
            'Seralyte is used as a key ingredient in the drug Cardivex.',
            'Cardivex was approved by the FDA for treating atrial fibrillation.',
            'NovaChem Inc. is headquartered in Basel, Switzerland.',
        ],
        'question': 'Where is the company headquartered that produces the key ingredient of the FDA-approved atrial fibrillation drug?',
        'answer_keywords': ['basel', 'switzerland'],
        'hops': 4,
    },
    {
        'facts': [
            'The Titan X satellite was launched by OrbitalDynamics Corp.',
            'Titan X orbits at an altitude of 550 kilometers.',
            'OrbitalDynamics Corp was founded by Dr. Priya Sharma.',
        ],
        'question': 'Who founded the company that launched the satellite orbiting at 550 km altitude?',
        'answer_keywords': ['priya sharma'],
        'hops': 3,
    },
]

print('\n🔗 Multi-Hop Reasoning over Long Context')
print('='*65)
hop_scores = []
for case in MULTI_HOP_CASES:
    paras = [random.choice(FILLER_PARAGRAPHS) for _ in range(15)]
    for i, fact in enumerate(case['facts']):
        insert_pos = int((i + 1) / (len(case['facts']) + 1) * len(paras))
        paras.insert(insert_pos, fact)
    context = '\n\n'.join(paras)
    prompt = f'{context}\n\nBased on the information above, answer: {case["question"]}\nAnswer concisely.'
    answer = groq_chat(prompt, model=GROQ_MODEL_SMART, max_tokens=80)
    answer_lower = answer.lower()
    hits = sum(1 for kw in case['answer_keywords'] if kw in answer_lower)
    score = hits / len(case['answer_keywords'])
    hop_scores.append(score)
    status = '✅' if score >= 0.5 else '❌'
    print(f'  {status} [{case["hops"]}-hop] Q: {case["question"][:60]}')
    print(f'     A: {answer[:80]}')
    print(f'     Score: {score:.2f}')
    print()

print(f'  Multi-hop accuracy: {np.mean(hop_scores):.2%}')


🔗 Multi-Hop Reasoning over Long Context
  ✅ [3-hop] Q: In which city does the 2022 Turing Award winner work?
     A: Portland, Oregon.
     Score: 1.00

  ✅ [4-hop] Q: Where is the company headquartered that produces the key ing
     A: Basel, Switzerland.
     Score: 1.00

  ✅ [3-hop] Q: Who founded the company that launched the satellite orbiting
     A: Dr. Priya Sharma.
     Score: 1.00

  Multi-hop accuracy: 100.00%


## 5. Counting & Enumeration Accuracy
### Tool: Groq — tests ability to count items scattered in long context

In [10]:
def build_counting_context(n_filler, items_to_count, item_name):
    """Build a context with specific items scattered among filler paragraphs."""
    paras = [random.choice(FILLER_PARAGRAPHS) for _ in range(n_filler)]
    item_sentences = [f'Note: A {item_name} was recorded at location #{i+1}.' for i in range(items_to_count)]
    # Scatter items throughout
    for item in item_sentences:
        pos = random.randint(0, len(paras))
        paras.insert(pos, item)
    return '\n\n'.join(paras)

counting_tests = [
    {'n_filler': 10, 'count': 3, 'item': 'blue crystal'},
    {'n_filler': 15, 'count': 5, 'item': 'red marker'},
    {'n_filler': 20, 'count': 7, 'item': 'signal beacon'},
    {'n_filler': 25, 'count': 10, 'item': 'data node'},
]

print('\n🔢 Counting & Enumeration Accuracy')
print('='*65)
count_results = []
for test in counting_tests:
    ctx = build_counting_context(test['n_filler'], test['count'], test['item'])
    prompt = f'{ctx}\n\nHow many times was a {test["item"]} mentioned or recorded in the text above? Reply with only the number.'
    try:
        answer = groq_chat(prompt, model=GROQ_MODEL_FAST, max_tokens=20)
        # Extract number from answer
        numbers = re.findall(r'\d+', answer)
        pred_count = int(numbers[0]) if numbers else -1
        correct = pred_count == test['count']
        error = abs(pred_count - test['count'])
    except Exception as e:
        answer = str(e)[:40]
        correct = False
        error = test['count']
        pred_count = -1
    
    count_results.append({'correct': correct, 'error': error, 'gold': test['count']})
    status = '✅' if correct else '❌'
    print(f'  {status}  Item="{test["item"]}"  Gold={test["count"]}  Pred={pred_count}  Error={error}  (in {test["n_filler"]} filler paras)')

count_acc = np.mean([r['correct'] for r in count_results])
avg_error = np.mean([r['error'] for r in count_results])
print(f'\n  Counting Accuracy: {count_acc:.1%}')
print(f'  Average Count Error: {avg_error:.1f}')


🔢 Counting & Enumeration Accuracy
  ✅  Item="blue crystal"  Gold=3  Pred=3  Error=0  (in 10 filler paras)
  ❌  Item="red marker"  Gold=5  Pred=6  Error=1  (in 15 filler paras)
  ❌  Item="signal beacon"  Gold=7  Pred=13  Error=6  (in 20 filler paras)
  ❌  Item="data node"  Gold=10  Pred=14  Error=4  (in 25 filler paras)

  Counting Accuracy: 25.0%
  Average Count Error: 2.8


## 6. Information Ordering & Chronological Understanding
### Tool: Groq — tests ability to reconstruct temporal order from scattered events

In [11]:
EVENTS_CHRONOLOGICAL = [
    ('1969', 'Apollo 11 moon landing'),
    ('1989', 'Fall of the Berlin Wall'),
    ('1991', 'Dissolution of the Soviet Union'),
    ('2001', 'Launch of Wikipedia'),
    ('2008', 'Global Financial Crisis'),
    ('2020', 'COVID-19 pandemic declared'),
]

def build_ordering_context(events, n_filler=15):
    """Scatter events in random order among filler paragraphs."""
    paras = [random.choice(FILLER_PARAGRAPHS) for _ in range(n_filler)]
    shuffled = events.copy()
    random.shuffle(shuffled)
    for year, event in shuffled:
        sentence = f'Historical note: In {year}, the following occurred: {event}.'
        pos = random.randint(0, len(paras))
        paras.insert(pos, sentence)
    return '\n\n'.join(paras)

ordering_tests = [
    {'events': EVENTS_CHRONOLOGICAL[:3], 'label': '3 events'},
    {'events': EVENTS_CHRONOLOGICAL[:5], 'label': '5 events'},
    {'events': EVENTS_CHRONOLOGICAL,     'label': '6 events'},
]

print('\n📅 Information Ordering (Chronological)')
print('='*65)
ordering_results = []
for test in ordering_tests:
    ctx = build_ordering_context(test['events'])
    events_list = ', '.join(e[1] for e in test['events'])
    prompt = f'{ctx}\n\nList the following events in chronological order (earliest first): {events_list}\nReturn them as a numbered list.'
    answer = groq_chat(prompt, model=GROQ_MODEL_SMART, max_tokens=200)
    
    # Check ordering by looking for years in sequence
    gold_order = [e[0] for e in test['events']]  # already chronological
    found_years = re.findall(r'(19\d{2}|20\d{2})', answer)
    
    # Calculate how many consecutive pairs are in correct order
    correct_pairs = 0
    total_pairs = len(gold_order) - 1
    for i in range(len(found_years) - 1):
        if found_years[i] <= found_years[i+1]:
            correct_pairs += 1
    
    pair_acc = correct_pairs / total_pairs if total_pairs > 0 else 0
    ordering_results.append(pair_acc)
    
    status = '✅' if pair_acc >= 0.8 else '⚠️' if pair_acc >= 0.5 else '❌'
    print(f'  {status} {test["label"]}: pair accuracy={pair_acc:.1%}')
    print(f'     Gold order: {gold_order}')
    print(f'     Found order: {found_years}')
    print()

print(f'  Avg Ordering Accuracy: {np.mean(ordering_results):.1%}')


📅 Information Ordering (Chronological)
  ✅ 3 events: pair accuracy=100.0%
     Gold order: ['1969', '1989', '1991']
     Found order: ['1969', '1989', '1991']

  ✅ 5 events: pair accuracy=100.0%
     Gold order: ['1969', '1989', '1991', '2001', '2008']
     Found order: ['1969', '1989', '1991', '2001', '2008']

  ✅ 6 events: pair accuracy=100.0%
     Gold order: ['1969', '1989', '1991', '2001', '2008', '2020']
     Found order: ['1969', '1989', '1991', '2001', '2008', '2020']

  Avg Ordering Accuracy: 100.0%


## 7. Cross-Document Comparison
### Tool: Groq — tests ability to compare and contrast information across documents

In [12]:
COMPARISON_CASES = [
    {
        'docs': [
            'Report A: The Falcon rocket achieved an orbital velocity of 7,800 m/s during its test flight. The mission cost was $62 million. The payload capacity was 22,800 kg to LEO.',
            'Report B: The Atlas rocket reached an orbital velocity of 7,600 m/s on its maiden voyage. The total mission cost was $109 million. It can carry 18,850 kg to LEO.',
        ],
        'questions': [
            {'q': 'Which rocket achieved higher orbital velocity?', 'answer_kw': ['falcon']},
            {'q': 'Which rocket was more expensive?', 'answer_kw': ['atlas']},
            {'q': 'Which rocket has greater payload capacity?', 'answer_kw': ['falcon']},
        ]
    },
    {
        'docs': [
            'Study X: Treatment Alpha reduced symptoms by 45% over 12 weeks in a trial of 500 patients. Side effects were reported by 12% of participants.',
            'Study Y: Treatment Beta reduced symptoms by 38% over 12 weeks in a trial of 800 patients. Side effects were reported by 5% of participants.',
        ],
        'questions': [
            {'q': 'Which treatment was more effective at reducing symptoms?', 'answer_kw': ['alpha']},
            {'q': 'Which treatment had fewer side effects?', 'answer_kw': ['beta']},
            {'q': 'Which study had more participants?', 'answer_kw': ['beta', 'study y', '800']},
        ]
    },
]

print('\n📊 Cross-Document Comparison')
print('='*65)
comparison_scores = []
for case in COMPARISON_CASES:
    # Scatter documents among filler
    paras = [random.choice(FILLER_PARAGRAPHS) for _ in range(10)]
    for doc in case['docs']:
        pos = random.randint(0, len(paras))
        paras.insert(pos, doc)
    ctx = '\n\n'.join(paras)
    
    print(f'\n  Documents: {len(case["docs"])}')
    for q_test in case['questions']:
        prompt = f'{ctx}\n\nBased on the documents above: {q_test["q"]}\nAnswer concisely.'
        answer = groq_chat(prompt, model=GROQ_MODEL_SMART, max_tokens=80)
        answer_lower = answer.lower()
        hit = any(kw in answer_lower for kw in q_test['answer_kw'])
        comparison_scores.append(1.0 if hit else 0.0)
        status = '✅' if hit else '❌'
        print(f'    {status} Q: {q_test["q"]}')
        print(f'       A: {answer[:70]}')

print(f'\n  Cross-Document Comparison Accuracy: {np.mean(comparison_scores):.1%}')


📊 Cross-Document Comparison

  Documents: 2
    ✅ Q: Which rocket achieved higher orbital velocity?
       A: The Falcon rocket achieved a higher orbital velocity of 7,800 m/s, com
    ✅ Q: Which rocket was more expensive?
       A: The Atlas rocket ($109 million) was more expensive than the Falcon roc
    ✅ Q: Which rocket has greater payload capacity?
       A: The Falcon rocket has a greater payload capacity (22,800 kg) compared 

  Documents: 2
    ✅ Q: Which treatment was more effective at reducing symptoms?
       A: Treatment Alpha (45% reduction) was more effective than Treatment Beta
    ✅ Q: Which treatment had fewer side effects?
       A: Treatment Beta had fewer side effects (5% vs 12%).
    ✅ Q: Which study had more participants?
       A: Study Y had more participants (800) than Study X (500).

  Cross-Document Comparison Accuracy: 100.0%


## 8. Long-Document Summarization Quality
### Tool: Sentence-Transformers + Groq judge

In [2]:
# pip install transformers==4.39.3 sentence-transformers==2.6.1

In [3]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

st_model = SentenceTransformer('all-MiniLM-L6-v2')

long_doc = '\n\n'.join(FILLER_PARAGRAPHS * 3)

SUMM_PROMPT = f'Summarize the following document in 3-4 sentences, covering the main topics discussed.\n\n{long_doc}\n\nSummary:'

summary = groq_chat(SUMM_PROMPT, model=GROQ_MODEL_FAST, max_tokens=200)

topic_sentences = FILLER_PARAGRAPHS
topic_embs = st_model.encode(topic_sentences, batch_size=32, show_progress_bar=False)
summary_emb  = st_model.encode([summary])
similarities = cosine_similarity(summary_emb, topic_embs)[0]
coverage_threshold = 0.35
covered = sum(1 for s in similarities if s > coverage_threshold)
coverage = covered / len(topic_sentences)
compression = len(summary.split()) / len(long_doc.split())

FAITH_SYS = '''You are a summarization evaluator. Score the summary on:
1. faithfulness (0.0-1.0): does it only contain info from the source?
2. coherence (0.0-1.0): is it well-organized and readable?
3. informativeness (0.0-1.0): does it capture the key topics?
Return JSON only: {"faithfulness": ..., "coherence": ..., "informativeness": ...}'''

faith_prompt = f'Source document (first 1500 chars):\n{long_doc[:1500]}\n\nSummary:\n{summary}'
faith_scores = parse_json(groq_chat(faith_prompt, system=FAITH_SYS))

print('\n📝 Long-Document Summarization Evaluation')
print('='*60)
print(f'  Summary: {summary[:150]}...')
print(f'\n  Topic Coverage:    {covered}/{len(topic_sentences)} topics ({coverage:.1%})')
print(f'  Compression Ratio: {compression:.2%}')
print(f'  Faithfulness:      {faith_scores.get("faithfulness", "N/A")}')
print(f'  Coherence:         {faith_scores.get("coherence", "N/A")}')
print(f'  Informativeness:   {faith_scores.get("informativeness", "N/A")}')

# Per-topic coverage detail
print(f'\n  Per-Topic Similarity to Summary:')
for i, (para, sim) in enumerate(zip(FILLER_PARAGRAPHS, similarities)):
    icon = "✅" if sim > coverage_threshold else "❌"
    topic = para[:50].strip()
    print(f'    {icon} {sim:.3f}  {topic}...')

/home/ubuntu/miniconda3/envs/llm-env/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/home/ubuntu/miniconda3/envs/llm-env/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(



📝 Long-Document Summarization Evaluation
  Summary: Here's a 4-sentence summary of the document covering the main topics discussed:

The document covers a wide range of topics, including the history of ...

  Topic Coverage:    1/10 topics (10.0%)
  Compression Ratio: 9.04%
  Faithfulness:      0.6
  Coherence:         0.8
  Informativeness:   0.7

  Per-Topic Similarity to Summary:
    ❌ 0.318  The history of mathematics spans thousands of year...
    ❌ 0.156  Ocean currents play a critical role in regulating...
    ✅ 0.421  The development of the printing press in the 15th...
    ❌ 0.141  Photosynthesis is the process by which plants conv...
    ❌ 0.282  The field of artificial intelligence has evolved s...
    ❌ 0.197  Volcanoes are geological formations created by the...
    ❌ 0.334  Classical music has undergone many transformations...
    ❌ 0.147  The human immune system is a complex network of ce...
    ❌ 0.302  Astronomy has captivated humans since ancient time...
    ❌ 0.287 

## 9. Token-Counting Awareness & Context Window Stress Test
### Tool: tiktoken + Groq

In [4]:
import tiktoken

enc = tiktoken.get_encoding('cl100k_base')

def count_tokens(text):
    return len(enc.encode(text))

print('\n📏 Context Length Analysis')
print('='*60)
for n_para in [10, 20, 40, 60, 80]:
    ctx = '\n\n'.join([random.choice(FILLER_PARAGRAPHS) for _ in range(n_para)])
    tokens = count_tokens(ctx)
    print(f'  {n_para:>3} paragraphs -> ~{tokens:>6,} tokens  ({len(ctx):>8,} chars)')

STRESS_NEEDLE = 'The activation code is ALPHA-7749-OMEGA.'
STRESS_Q = 'What is the activation code mentioned in the text? Reply with only the code.'

stress_sizes = [10, 20, 40, 60, 80]
print(f'\n🏋️  Context Window Stress Test')
print('='*60)
stress_results = []
for n in stress_sizes:
    paras = [random.choice(FILLER_PARAGRAPHS) for _ in range(n)]
    paras.insert(n // 2, STRESS_NEEDLE)
    ctx = '\n\n'.join(paras)
    tokens = count_tokens(ctx)
    try:
        answer = groq_chat(f'{ctx}\n\n{STRESS_Q}', model=GROQ_MODEL_FAST, max_tokens=30)
        hit = 'alpha-7749-omega' in answer.lower().replace(' ', '')
        status = '✅' if hit else '❌'
    except Exception as e:
        answer = str(e)[:50]
        hit = False
        status = '💥'
    stress_results.append({'n': n, 'tokens': tokens, 'hit': hit})
    print(f'  {status} {n:>3} paras (~{tokens:>5,} tok)  A: {answer[:50]}')

stress_acc = np.mean([r['hit'] for r in stress_results])
print(f'\n  Stress Test Accuracy: {stress_acc:.1%}')


📏 Context Length Analysis
   10 paragraphs -> ~   483 tokens  (   2,648 chars)
   20 paragraphs -> ~   959 tokens  (   5,345 chars)
   40 paragraphs -> ~ 1,927 tokens  (  10,722 chars)
   60 paragraphs -> ~ 2,880 tokens  (  16,138 chars)
   80 paragraphs -> ~ 3,858 tokens  (  21,399 chars)

🏋️  Context Window Stress Test
  ✅  10 paras (~  499 tok)  A: ALPHA-7749-OMEGA.
  ✅  20 paras (~  956 tok)  A: ALPHA-7749-OMEGA.
  ✅  40 paras (~1,964 tok)  A: ALPHA-7749-OMEGA.
  ✅  60 paras (~2,905 tok)  A: ALPHA-7749-OMEGA.
  ✅  80 paras (~3,940 tok)  A: ALPHA-7749-OMEGA

  Stress Test Accuracy: 100.0%


## 10. Aggregated Long-Context Scorecard

In [13]:
niah_avg  = np.mean([r['score'] for r in results_niah])
kv_avg    = np.mean([r['accuracy'] for r in kv_results])
pos_avg   = np.mean([r['score'] for r in pos_results])
hop_avg   = np.mean(hop_scores)

show('Long-Context Evaluation Scorecard', {
    'Needle-in-a-Haystack (NIAH)':      niah_avg,
    'Key-Value Retrieval':              kv_avg,
    'Position Bias (avg accuracy)':     pos_avg,
    'Position Bias (edge-mid diff)':    bias,
    'Multi-Hop Reasoning':              hop_avg,
    'Counting Accuracy':                count_acc,
    'Avg Count Error':                  avg_error,
    'Ordering Accuracy':                float(np.mean(ordering_results)),
    'Cross-Doc Comparison':             float(np.mean(comparison_scores)),
    'Summarization Coverage':           coverage,
    'Summarization Faithfulness':       float(faith_scores.get('faithfulness', 0)),
    'Stress Test Accuracy':             stress_acc,
})


  📊 Long-Context Evaluation Scorecard
  Needle-in-a-Haystack (NIAH)   : 1.0000
  Key-Value Retrieval           : 1.0000
  Position Bias (avg accuracy)  : 1.0000
  Position Bias (edge-mid diff) : 0.0000
  Multi-Hop Reasoning           : 1.0000
  Counting Accuracy             : 0.2500
  Avg Count Error               : 2.7500
  Ordering Accuracy             : 1.0000
  Cross-Doc Comparison          : 1.0000
  Summarization Coverage        : 0.1000
  Summarization Faithfulness    : 0.6000
  Stress Test Accuracy          : 1.0000
